# NYISO Solar Forecasting: Physics-Informed Feature Engineering for Residual Learning

## Imports and Configuration

In [24]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [25]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_NYISOSolarForecast"

data_root = repo_root / "data"
processed_dir = data_root / "processed"

merged_out = processed_dir / "03_merged_data.csv"

df = pd.read_csv(merged_out, low_memory=False)

## Standardize Columns 

In [26]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df["time_stamp"] = pd.to_datetime(df["time_stamp"], utc=True, errors="coerce")

if "time" in df.columns:
    df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
    same_time_mask = (df["time"] == df["time_stamp"]) | (
        df["time"].isna() & df["time_stamp"].isna()
    )
    if bool(same_time_mask.all()):
        df = df.drop(columns=["time"])

numeric_cols = [
    "actual_mw",
    "forecast_mw",
    "capacity_mw",
    "temperature_2m",
    "surface_pressure",
    "cloud_cover",
    "windspeed_10m",
    "shortwave_radiation",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["zone_name"] = df["zone_name"].astype(str).str.strip().str.upper()

## Time and Error Metric Features

In [27]:
df["time_local"] = df["time_stamp"].dt.tz_convert("America/New_York")


df["year_from_ts"] = df["time_local"].dt.year
df["month_local"] = df["time_local"].dt.month
df["dayofweek_local"] = df["time_local"].dt.dayofweek
df["hour_local"] = df["time_local"].dt.hour

df["is_weekend"] = df["dayofweek_local"].isin([5, 6]).astype(int)
df["is_daylight_proxy"] = (df["shortwave_radiation"] > 0).astype(int)

In [28]:
df["forecast_error_mw"] = df["actual_mw"] - df["forecast_mw"]
df["absolute_error_mw"] = (df["actual_mw"] - df["forecast_mw"]).abs()
df["smape"] = np.where(
    (df["actual_mw"].abs() + df["forecast_mw"].abs()) > 0,
    200 * df["absolute_error_mw"] / (df["actual_mw"].abs() + df["forecast_mw"].abs()),
    np.nan
)

## SYSTEM-Only Modeling Table

In [29]:
df_system = (
    df[df["zone_name"] == "SYSTEM"]
    .copy()
    .sort_values("time_stamp")
    .reset_index(drop=True)
)

print("System Shape:", df_system.shape)
print("System Time Range:", df_system["time_stamp"].min(), "to", df_system["time_stamp"].max())
print("System Columns:", df_system.columns.tolist())

System Shape: (42455, 21)
System Time Range: 2020-11-17 05:00:00+00:00 to 2025-09-21 03:00:00+00:00
System Columns: ['time_stamp', 'zone_name', 'actual_mw', 'forecast_mw', 'capacity_mw', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'year', 'time_local', 'year_from_ts', 'month_local', 'dayofweek_local', 'hour_local', 'is_weekend', 'is_daylight_proxy', 'forecast_error_mw', 'absolute_error_mw', 'smape']


### Encoding Cyclic Time 
solar gen is periodic with 24 hours and 365 days, sin(theta) = sin((2*pi*t)/T)

In [30]:
df_system["dayofyear_local"] = df_system["time_local"].dt.dayofyear
df_system["weekofyear_local"] = df_system["time_local"].dt.isocalendar().week.astype(int)

df_system["hour_sin"] = np.sin(2 * np.pi * df_system["hour_local"] / 24)
df_system["hour_cos"] = np.cos(2 * np.pi * df_system["hour_local"] / 24)

df_system["month_sin"] = np.sin(2 * np.pi * df_system["month_local"] / 12)
df_system["month_cos"] = np.cos(2 * np.pi * df_system["month_local"] / 12)

df_system["dayofyear_sin"] = np.sin(2 * np.pi * df_system["dayofyear_local"] / 365.25)
df_system["dayofyear_cos"] = np.cos(2 * np.pi * df_system["dayofyear_local"] / 365.25)

df_system["dayofweek_sin"] = np.sin(2 * np.pi * df_system["dayofweek_local"] / 7)
df_system["dayofweek_cos"] = np.cos(2 * np.pi * df_system["dayofweek_local"] / 7)

### Solar Regimes

In [31]:
df_system["is_morning_ramp"] = df_system["hour_local"].between(6, 9).astype(int)
df_system["is_midday"] = df_system["hour_local"].between(10, 14).astype(int)
df_system["is_evening_ramp"] = df_system["hour_local"].between(15, 18).astype(int)
df_system["is_night"] = (df_system["is_daylight_proxy"] == 0).astype(int)

### Understanding Capacity 
capacity ratio is forecast/installed cap

In [32]:
df_system["capacity_ffill"] = df_system["capacity_mw"].ffill().bfill()

df_system["forecast_capacity_ratio"] = np.where(
    df_system["capacity_ffill"] > 0,
    df_system["forecast_mw"] / df_system["capacity_ffill"],
    np.nan
)

### Physics-Informed Features Interactions
PV depends on irradiance, temp, and how clouds move
P= f(G,T,C)

In [33]:
df_system["cloud_cover_frac"] = df_system["cloud_cover"] / 100.0

df_system["forecast_mw_log1p"] = np.log1p(df_system["forecast_mw"].clip(lower=0))
df_system["shortwave_radiation_log1p"] = np.log1p(df_system["shortwave_radiation"].clip(lower=0))

In [34]:
interaction_features = pd.DataFrame(index=df_system.index)

interaction_features["forecast_x_hour_sin"] = df_system["forecast_mw"] * df_system["hour_sin"]
interaction_features["forecast_x_hour_cos"] = df_system["forecast_mw"] * df_system["hour_cos"]

interaction_features["shortwave_x_cloud"] = (
    df_system["shortwave_radiation"] * df_system["cloud_cover_frac"]
)

interaction_features["shortwave_x_temp"] = (
    df_system["shortwave_radiation"] * df_system["temperature_2m"]
)

interaction_features["temp_x_wind"] = (
    df_system["temperature_2m"] * df_system["windspeed_10m"]
)

df_system = pd.concat([df_system, interaction_features], axis=1)

### Rolling Windows and Ramp Features

In [35]:
rolling_windows = [3, 6, 24]

rolling_features = {}

for w in rolling_windows:

    rolling_features[f"forecast_roll_mean_{w}"] = (
        df_system["forecast_mw"].shift(1).rolling(w, min_periods=1).mean()
    )

    rolling_features[f"shortwave_roll_mean_{w}"] = (
        df_system["shortwave_radiation"].shift(1).rolling(w, min_periods=1).mean()
    )

rolling_df = pd.DataFrame(rolling_features)

df_system = pd.concat([df_system, rolling_df], axis=1)

In [36]:
df_system["forecast_diff_1"] = df_system["forecast_mw"].diff(1)
df_system["shortwave_diff_1"] = df_system["shortwave_radiation"].diff(1)

df_system["forecast_ramp_abs"] = df_system["forecast_diff_1"].abs()
df_system["shortwave_ramp_abs"] = df_system["shortwave_diff_1"].abs()

## Missingness Summary

In [37]:
feature_missing_summary = (
    pd.DataFrame({
        "column": df_system.columns,
        "dtype": [str(df_system[c].dtype) for c in df_system.columns],
        "missing_count": [df_system[c].isna().sum() for c in df_system.columns],
    })
    .assign(missing_pct=lambda x: 100 * x["missing_count"] / len(df_system))
    .sort_values(["missing_count", "column"], ascending=[False, True])
)

feature_missing_summary.head(40)

,column,dtype,missing_count,missing_pct
36,forecast_capacity_ratio,float64,26504,62.428454
4,capacity_mw,float64,23903,56.301967
20,smape,float64,18910,44.541279
19,absolute_error_mw,float64,1000,2.355435
18,forecast_error_mw,float64,1000,2.355435
2,actual_mw,float64,712,1.677070
51,forecast_diff_1,float64,301,0.708986
53,forecast_ramp_abs,float64,301,0.708986
3,forecast_mw,float64,288,0.678365
38,forecast_mw_log1p,float64,288,0.678365


## Modeling Dataset
The target is the residual, the error. Also columns that could cause leakage needs to be dropped.

In [48]:
target = "forecast_error_mw"

drop_cols = [
    "actual_mw",
    "absolute_error_mw",
    "smape",
    "time_stamp",
    "time_local",
    "date_local",
    "zone_name"
]

feature_cols = [c for c in df_system.columns if c not in drop_cols + [target]]

X = df_system[feature_cols]
y = df_system[target]

ValueError: The column label 'time_stamp' is not unique.

## Train/Test Split 

In [43]:
split_date = "2024-01-01"

train_mask = df_system["time_stamp"] < split_date

X_train = X.loc[train_mask]
X_test = X.loc[~train_mask]

y_train = y.loc[train_mask]
y_test = y.loc[~train_mask]

## Baselines

In [46]:
## Results and Interpretations